<a href="https://colab.research.google.com/github/Sreejani17/Movie_Review_Sentiment_Analysis/blob/main/Movie_Review_Sentiment_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎬 Movie Review Sentiment Analysis (NLP)

A machine learning project that classifies movie reviews as positive or
negative, using text data and comparing two classification algorithms.

**Goal:** Predict sentiment (positive/negative) from raw review text.

**Dataset:** IMDB Dataset of 50K Movie Reviews (Kaggle)

**Tech stack:** Python, pandas, scikit-learn (TF-IDF, Logistic Regression, Naive Bayes)

**Author:** [Sreejani Parampogu] | [https://github.com/Sreejani17]

## 1. Load the Dataset
We use the IMDB dataset from Kaggle, which contains 50,000 movie reviews
labeled as positive or negative.

In [17]:
from google.colab import files
uploaded = files.upload()

Saving IMDB Dataset.csv to IMDB Dataset (1).csv


In [18]:
import pandas as pd

df = pd.read_csv('IMDB Dataset.csv')
df.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


## 2. Explore the Data
We check the dataset's size, structure, and class balance before doing
any cleaning or modeling.

In [19]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   review     50000 non-null  object
 1   sentiment  50000 non-null  object
dtypes: object(2)
memory usage: 781.4+ KB


,0
review,0
sentiment,0


In [20]:
df['sentiment'].value_counts()

,count
sentiment,
positive,25000
negative,25000


## 3. Clean the Text
Raw review text contains HTML tags, punctuation, and mixed casing — none
of which help the model learn sentiment. We clean it so only meaningful
words remain.

In [21]:
import re

def clean_text(text):
    text = text.lower()                          # lowercase everything
    text = re.sub(r'<.*?>', '', text)             # remove HTML tags like <br>
    text = re.sub(r'[^a-z\s]', '', text)          # remove punctuation/numbers
    return text

df['clean_review'] = df['review'].apply(clean_text)
df[['review', 'clean_review']].head()

,review,clean_review
0,One of the other reviewers has mentioned that ...,one of the other reviewers has mentioned that ...
1,A wonderful little production. <br /><br />The...,a wonderful little production the filming tech...
2,I thought this was a wonderful way to spend ti...,i thought this was a wonderful way to spend ti...
3,Basically there's a family where a little boy ...,basically theres a family where a little boy j...
4,"Petter Mattei's ""Love in the Time of Money"" is...",petter matteis love in the time of money is a ...


## 4. Convert Labels to Numbers
Models need numeric targets, so we map sentiment text to 0/1.

In [22]:
df['sentiment_num'] = df['sentiment'].map({'positive': 1, 'negative': 0})

## 5. Split into Training and Test Sets
80% of reviews are used to train the model, 20% are held back to test
it on reviews it has never seen before.

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_review'], df['sentiment_num'], test_size=0.2, random_state=42
)

## 6. Convert Text to Numbers (TF-IDF)
Machine learning models can't read text directly. TF-IDF scores each word
by how important/unique it is across all reviews, turning text into
numeric features.

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

## 7. Train Two Models and Compare
We train Logistic Regression and Naive Bayes — two different algorithms
commonly used for text classification — and compare their accuracy.

In [25]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

# Model 1: Logistic Regression
log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_tfidf, y_train)
log_pred = log_model.predict(X_test_tfidf)

# Model 2: Naive Bayes
nb_model = MultinomialNB()
nb_model.fit(X_train_tfidf, y_train)
nb_pred = nb_model.predict(X_test_tfidf)

print("Logistic Regression Accuracy:", accuracy_score(y_test, log_pred))
print("Naive Bayes Accuracy:", accuracy_score(y_test, nb_pred))

Logistic Regression Accuracy: 0.8835
Naive Bayes Accuracy: 0.8477


## 8. Compare Results Side-by-Side

In [26]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression', 'Naive Bayes'],
    'Accuracy': [accuracy_score(y_test, log_pred), accuracy_score(y_test, nb_pred)]
})
comparison

,Model,Accuracy
0,Logistic Regression,0.8835
1,Naive Bayes,0.8477


## 9. Detailed Evaluation (Best Model)
Accuracy alone doesn't tell the full story. The confusion matrix and
classification report show exactly how many positive/negative reviews
were correctly vs incorrectly classified.

In [27]:
from sklearn.metrics import confusion_matrix, classification_report

print("Confusion Matrix:\n", confusion_matrix(y_test, log_pred))
print("\nClassification Report:\n", classification_report(y_test, log_pred))

Confusion Matrix:
 [[4296  665]
 [ 500 4539]]

Classification Report:
               precision    recall  f1-score   support

           0       0.90      0.87      0.88      4961
           1       0.87      0.90      0.89      5039

    accuracy                           0.88     10000
   macro avg       0.88      0.88      0.88     10000
weighted avg       0.88      0.88      0.88     10000



## 10. Test on a Custom Sentence

In [28]:
sample = ["This movie was absolutely wonderful, great acting and story!"]
sample_clean = [clean_text(sample[0])]
sample_tfidf = vectorizer.transform(sample_clean)

prediction = log_model.predict(sample_tfidf)
print("Positive" if prediction[0] == 1 else "Negative")

Positive


## 11. Conclusion

- Logistic Regression Accuracy: **[0.8835]**
- Naive Bayes Accuracy: **[0.8477]**
- Logistic Regression performed better, likely because it can weigh
  words more flexibly, while Naive Bayes assumes word occurrences are
  independent of each other, which is a simplifying assumption.
